# Importing libraries

In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
%matplotlib inline

In [35]:
import numpy as np
import pandas as pd
import math
import dill
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from src.transforms import add_date_columns, change_columns_from_str_to_categorical, process_df, split_vals
from src.plot import draw_tree

# Loading Datasets

### Kaggle S&P500 data
- | `sp500_stocks.csv.xz` | Daily open, high, low, close, adjusted close and volume for each stock |
- | `sp500_companies.csv` | One row per company: sector, industry, market cap, etc. |

In [36]:
df_stocks=pd.read_csv(f"data/sp500_stocks.csv.xz",parse_dates=['date'])
df_comp=pd.read_csv(f"data/sp500_companies.csv",parse_dates=['date_added'])

### FRED data
- | `VIXCLS` | VIX: the volatility the market expects over the next 30 days |
- | `DGS10` | 10-year US Treasury interest rate |
- | `T10Y2Y` | Difference between 10-year and 2-year rates (a classic recession signal) |

In [37]:
df_VIX=pd.read_csv(f"data/VIXCLS.csv",parse_dates=['observation_date'])
df_DGS=pd.read_csv(f"data/DGS10.csv",parse_dates=['observation_date'])
df_T10Y2Y=pd.read_csv(f"data/T10Y2Y.csv",parse_dates=['observation_date'])

# Presenting my data 

In [38]:
dfs = {'Stocks': df_stocks, 'Companies': df_comp, 'VIX': df_VIX,
       '10Y rate': df_DGS, '10Y-2Y spread': df_T10Y2Y}

for name, df in dfs.items():
    print(f"\n===== {name}: {df.shape[0]:,} rows x {df.shape[1]} columns =====")
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isnull().sum().sum())
    display(df.head(3))


===== Stocks: 2,949,733 rows x 7 columns =====
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol']
Missing values: 0


,date,open,high,low,close,volume,symbol
0,2000-01-03,46.873742,46.985347,40.103091,42.855995,4674353.0,A
1,2000-01-04,40.549510,40.995929,38.540636,39.582275,4765083.0,A
2,2000-01-05,39.433469,39.582277,35.899340,37.126984,5758642.0,A



===== Companies: 503 rows x 7 columns =====
Columns: ['symbol', 'company', 'sector', 'sub_industry', 'headquarters', 'date_added', 'founded']
Missing values: 0


,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888



===== VIX: 6,979 rows x 2 columns =====
Columns: ['observation_date', 'VIXCLS']
Missing values: 218


,observation_date,VIXCLS
0,2000-01-03,24.21
1,2000-01-04,27.01
2,2000-01-05,26.41



===== 10Y rate: 6,979 rows x 2 columns =====
Columns: ['observation_date', 'DGS10']
Missing values: 288


,observation_date,DGS10
0,2000-01-03,6.58
1,2000-01-04,6.49
2,2000-01-05,6.62



===== 10Y-2Y spread: 6,980 rows x 2 columns =====
Columns: ['observation_date', 'T10Y2Y']
Missing values: 288


,observation_date,T10Y2Y
0,2000-01-03,0.20
1,2000-01-04,0.19
2,2000-01-05,0.24


# Combining Datasets

In [39]:
comp = df_comp[['symbol', 'sector', 'company', 'sub_industry']]
vix  = df_VIX[['observation_date', 'VIXCLS']]
dgs  = df_DGS[['observation_date', 'DGS10']]
t10  = df_T10Y2Y[['observation_date', 'T10Y2Y']]

df_raw = df_stocks.merge(comp, on='symbol', how='left')

for fred in [vix, dgs, t10]:
    df_raw = df_raw.merge(fred, left_on='date', right_on='observation_date', how='left')
    df_raw = df_raw.drop(columns='observation_date')

In [40]:
print(f"\n===== Full Raw Dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns =====")
print("Columns:", df_raw.columns.tolist())
print("Missing values:", df_raw.isnull().sum().sum())
display(df_raw.head(3))


===== Full Raw Dataset: 2,949,733 rows x 13 columns =====
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'sector', 'company', 'sub_industry', 'VIXCLS', 'DGS10', 'T10Y2Y']
Missing values: 43164


,date,open,high,low,close,volume,symbol,sector,company,sub_industry,VIXCLS,DGS10,T10Y2Y
0,2000-01-03,46.873742,46.985347,40.103091,42.855995,4674353.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,24.21,6.58,0.20
1,2000-01-04,40.549510,40.995929,38.540636,39.582275,4765083.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,27.01,6.49,0.19
2,2000-01-05,39.433469,39.582277,35.899340,37.126984,5758642.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,26.41,6.62,0.24


In [41]:
%time df_raw.to_csv(f"data/full_dataset.csv", index = False)

CPU times: user 26.4 s, sys: 1.36 s, total: 27.7 s
Wall time: 29.7 s


# Data Preprocessing

## Handling missing values

The merged dataset has **2,949,733 rows × 13 columns** and **43,164 missing values** (about 0.1% of all cells).

Different columns need different strategies, because the missing values don't all mean the same thing:

| Column type | Columns | Strategy |
|---|---|---|
| Macro indicators | `VIXCLS`, `DGS10`, `T10Y2Y` | Forward-fill by date |
| Prices | `open`, `high`, `low`, `close`, `volume` | Drop or fill within each stock |
| Categories | `sector`, `company`, `sub_industry` | Recover from the stock's other rows |


In [42]:
df_raw = df_raw.sort_values(['symbol', 'date']).reset_index(drop=True)

print(df_raw.isna().sum())
print(df_raw[df_raw['DGS10'].isna()]['date'].dt.date.unique()[:20])
print(df_raw[df_raw['close'].isna()]['symbol'].value_counts().head())

date                0
open                0
high                0
low                 0
close               0
volume              0
symbol              0
sector              0
company             0
sub_industry        0
VIXCLS            503
DGS10           21582
T10Y2Y          21079
dtype: int64
[datetime.date(2000, 10, 9) datetime.date(2001, 10, 8)
 datetime.date(2001, 11, 12) datetime.date(2002, 10, 14)
 datetime.date(2002, 11, 11) datetime.date(2003, 10, 13)
 datetime.date(2003, 11, 11) datetime.date(2004, 10, 11)
 datetime.date(2004, 11, 11) datetime.date(2005, 10, 10)
 datetime.date(2005, 11, 11) datetime.date(2006, 10, 9)
 datetime.date(2007, 10, 8) datetime.date(2007, 11, 12)
 datetime.date(2008, 10, 13) datetime.date(2008, 11, 11)
 datetime.date(2009, 10, 12) datetime.date(2009, 11, 11)
 datetime.date(2010, 10, 11) datetime.date(2010, 11, 11)]
Series([], Name: count, dtype: int64)


## Diagnostic results

| Column | Missing | Cause |
|---|---|---|
| `DGS10` | 21,582 | Bond-market holidays (Columbus Day, Veterans Day): stocks trade, Treasuries don't |
| `T10Y2Y` | 21,079 | Same holidays |
| `VIXCLS` | 503 | A few days where the VIX was not published |
| All price & category columns | 0 | Complete |

The missing values are **whole dates**, not individual stocks: on a bond holiday, every stock is missing the yield.
So we only need to fix the 3 macro columns, by **forward-filling by date**: on a holiday, we use the previous trading day's rate.

## Macro indicators: forward-fill by date

`VIXCLS`, `DGS10` and `T10Y2Y` have **one value per day**, shared by every stock. So we:

1. Build a small table with one row per date.
2. Fill each gap with the **last known value** (`ffill`). On a bond holiday, the best estimate of the rate is the previous day's rate.
3. Merge the filled values back into the main dataset.

In [43]:
macro_cols = ['VIXCLS', 'DGS10', 'T10Y2Y']

macro = (df_raw.groupby('date')[macro_cols].first()
               .sort_index()
               .ffill())

df = df_raw.drop(columns=macro_cols).merge(macro, on='date', how='left')
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

assert df.isna().sum().sum() == 0, "There are still missing values!"
print("No missing values ✅")
print(f"Rows before: {len(df_raw):,} | Rows after: {len(df):,}")

No missing values ✅
Rows before: 2,949,733 | Rows after: 2,949,733


In [44]:
%time df.to_csv(f"data/full_dataset.csv", index = False)

CPU times: user 26.6 s, sys: 1.2 s, total: 27.8 s
Wall time: 29 s


In [45]:
print(f"\n===== Full Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns =====")
print("Columns:", df.columns.tolist())
print("Missing values:", df.isnull().sum().sum())
display(df.head(3))


===== Full Dataset: 2,949,733 rows x 13 columns =====
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'sector', 'company', 'sub_industry', 'VIXCLS', 'DGS10', 'T10Y2Y']
Missing values: 0


,date,open,high,low,close,volume,symbol,sector,company,sub_industry,VIXCLS,DGS10,T10Y2Y
0,2000-01-03,46.873742,46.985347,40.103091,42.855995,4674353.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,24.21,6.58,0.20
1,2000-01-04,40.549510,40.995929,38.540636,39.582275,4765083.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,27.01,6.49,0.19
2,2000-01-05,39.433469,39.582277,35.899340,37.126984,5758642.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,26.41,6.62,0.24



# Tree fitting

## 1. The variable to predict: next-week realized volatility

Following the project objective, we predict **how much each stock will move over the next 5 trading days**.

For a stock at the end of week $t$, with $r_{t+k}$ its daily log return $k$ days later, the **realized volatility** of the next week is:

$$ RV_{t \to t+5} = \sqrt{\sum_{k=1}^{5} r_{t+k}^2} $$

We predict $\log(RV)$ rather than $RV$:
- volatility is very skewed, and the log makes the distribution symmetric;
- an error in log space is a **relative** error: being 20% wrong counts the same for a calm stock and for a turbulent one.

In [46]:
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
df['log_close'] = np.log(df['close'])
g = df.groupby('symbol')

df['ret'] = g['log_close'].diff()
df['ret_sq'] = df['ret'] ** 2
g = df.groupby('symbol')

df['rv_next5'] = np.sqrt(sum(g['ret_sq'].shift(-k) for k in range(1, 6)))

## 2. Stock-level features

All features use **only past data**, computed **per stock** (`groupby('symbol')`) so that one stock never leaks into another.

| Feature | Meaning | Why it should help |
|---|---|---|
| `rv_5d`, `rv_20d`, `rv_60d` | realized volatility over the last week / month / quarter (rescaled to a weekly horizon) | volatility clusters: recent volatility predicts future volatility |
| `range_5d` | average daily high-low range over the week | captures intraday turbulence that closing prices miss |
| `ret_5d`, `ret_20d` | past weekly / monthly return | big drops tend to be followed by higher volatility (*leverage effect*) |
| `volume_z` | this week's trading volume vs its 3-month average | unusual activity often comes with news and turbulence |

`rv_5d` is also what the **persistence baseline** uses: "next week will be as volatile as this week".


In [47]:
def past_rv(k):
    return np.sqrt(g['ret_sq'].transform(lambda s: s.rolling(k).mean()) * 5)

df['rv_5d']  = past_rv(5)   
df['rv_20d'] = past_rv(20) 
df['rv_60d'] = past_rv(60)  

df['hl_range'] = np.log(df['high'] / df['low'])
df['range_5d'] = g['hl_range'].transform(lambda s: s.rolling(5).mean())

df['ret_5d']  = g['log_close'].diff(5)
df['ret_20d'] = g['log_close'].diff(20)

df['log_volume'] = np.log1p(df['volume'])
df['volume_z'] = (g['log_volume'].transform(lambda s: s.rolling(5).mean())
                  - g['log_volume'].transform(lambda s: s.rolling(60).mean()))